# s09 — Leiden clustering of the FFP output adjacency matrix

Bipartite Leiden clustering of the FFP -> central-brain output adjacency matrix
produced by `Data_Processing/s08_FFP_ADJ_matrix.m`. Reads from `Processed_Data/`:
- `right_FFP_output_matrix_CB_no_VPN_thr0.csv` — FFP (rows) x post-neuron (cols) synapse matrix
- `right_FFP_table.csv` — FFP neuron table (rows)
- `post_right_FFP_CB_no_VPN_thr0.csv` — post-neuron root_ids (cols)

Writes the cluster assignments, inter-cluster connectivity, dendrogram, and the
edge-switching modularity null distribution back to `Processed_Data/`.

Requires the `scikit-network` (sknetwork) package.

In [ ]:
from IPython.display import SVG

import os
import numpy as np
from scipy import sparse
import pandas as pd
from sknetwork.visualization import visualize_graph, visualize_bigraph, visualize_dendrogram
from sknetwork.clustering import Leiden, get_modularity
from sknetwork.hierarchy import Paris

# Base folder holding the s08 input matrices and the clustering outputs.
# baseDir is resolved from the notebook's location (run from Data_Processing/),
# so the code runs after download without editing any path.
baseDir = os.path.dirname(os.getcwd())
processed_dir = os.path.join(baseDir, "Processed_Data")

In [ ]:
import os

# FFP -> central-brain output adjacency matrix (rows = FFP, cols = post neurons)
ADJ = np.loadtxt(os.path.join(processed_dir, "right_FFP_output_matrix_CB_no_VPN_thr0.csv"),
                 delimiter=",", dtype=float)

In [ ]:
#ADJ=np.transpose(ADJ)
ADJ.shape

In [ ]:
biadjacency = sparse.csr_matrix(ADJ)

In [ ]:
def run_leiden_repeatedly(biadjacency, num_iterations,resolution=1):
    best_modularity = -np.inf
    best_labels = None

    for _ in range(num_iterations):
        # Leiden clustering
        leiden = Leiden(resolution=resolution,shuffle_nodes=True,sort_clusters=True)
        leiden.fit(biadjacency)

        # Separate row / column cluster labels
        labels_row = leiden.labels_row_
        labels_col = leiden.labels_col_

        # Compute modularity
        modularity = get_modularity(biadjacency, labels_row, labels_col)

        # Keep the result with the highest modularity
        if modularity > best_modularity:
            best_modularity = modularity
            best_labels_row = labels_row
            best_labels_col = labels_col
            best_leiden=leiden

    return best_modularity, best_labels_row, best_labels_col,best_leiden

In [ ]:
# Run repeatedly and keep the best partition
num_iterations = 100000
resolution=3
best_modularity, best_labels_row,best_labels_col,best_leiden  = run_leiden_repeatedly(biadjacency, num_iterations,resolution=resolution)

print(f"Best Modularity: {best_modularity}")

labels_row_unique, counts_row = np.unique(best_labels_row, return_counts=True)
print(labels_row_unique, counts_row)

labels_col_unique, counts_col = np.unique(best_labels_col, return_counts=True)
print(labels_col_unique, counts_col)

In [ ]:
biadjacency_aggregate = best_leiden.aggregate_


In [ ]:
biadjacency_aggregate

In [ ]:
# FFP neuron table (rows of ADJ)
Data = pd.read_csv(os.path.join(processed_dir, "right_FFP_table.csv"))

In [ ]:
Data['Cluster']=None

In [ ]:
Data['Cluster']=best_labels_row


In [ ]:
Data.to_csv(os.path.join(processed_dir, 'leiden_right_FFP_output_CB_no_VPN_thr0_resol3_100000.csv'))

In [ ]:
biadjacency_aggregate_matrix=biadjacency_aggregate.toarray()

In [ ]:
biadjacency_aggregate_matrix

In [ ]:
np.shape(biadjacency_aggregate_matrix)

In [ ]:
np.savetxt(os.path.join(processed_dir, 'leiden_right_FFP_intercluster_connectivity_CB_no_VPN_thr0_resol3_100000.csv'), biadjacency_aggregate_matrix, delimiter=',')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
from sknetwork.hierarchy import Paris
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt
B=biadjacency_aggregate_matrix;
# Compute similarity matrices


In [ ]:
np.shape(B)

In [ ]:
from scipy.sparse import csr_matrix
from sknetwork.hierarchy import Paris
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt

# --- Raw similarity ---
S = B @ B.T

# Paris (raw)
paris_raw = Paris()
paris_raw.fit(csr_matrix(S))
Z_raw = paris_raw.dendrogram_

# Plot (raw only)
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
dendrogram(Z_raw, ax=ax, no_labels=True)
ax.set_title('Dendrogram (Raw Similarity)')
plt.tight_layout()
plt.show()


In [ ]:
from scipy.sparse import csr_matrix
from sknetwork.hierarchy import Paris
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt
import numpy as np

# --- Raw similarity ---
S = B @ B.T

# Paris (raw)
paris_raw = Paris()
paris_raw.fit(csr_matrix(S))
Z_raw = paris_raw.dendrogram_.astype(float).copy()

# --- Replace Inf/NaN distances with a large finite value ---
mask_bad = ~np.isfinite(Z_raw[:, 2])
if mask_bad.any():
    BIG = 1e12  # very large value (adjust e.g. 1e9~1e15 if needed)
    Z_raw[mask_bad, 2] = BIG

# Plot (raw only)
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
dendrogram(Z_raw, ax=ax, no_labels=True)
ax.set_title('Dendrogram (Raw Similarity)')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Convert to DataFrame
df_raw = pd.DataFrame(Z_raw, columns=['cluster1', 'cluster2', 'distance', 'size'])

# Save to CSV
df_raw.to_csv(os.path.join(processed_dir, 'dendrogram_raw_CB_no_VPN_thr0_resol3_100000.csv'), index=False)

In [ ]:
# Post-neuron root_ids (columns of ADJ)
PreNeuron = pd.read_csv(os.path.join(processed_dir, "post_right_FFP_CB_no_VPN_thr0.csv"), header=None)

In [ ]:
PreNeuron['Cluster']=None

In [ ]:
PreNeuron['Cluster']=best_labels_col


In [ ]:
PreNeuron.to_csv(os.path.join(processed_dir, 'leiden_post_right_FFP_CB_no_VPN_thr0_100000.csv'))

In [ ]:
get_modularity(biadjacency,labels=best_labels_row,labels_col=best_labels_col)

In [ ]:
biadjacency